# Stage 23A: strong-base aligned physical offset-state audit

Stage 22のrowwise residual fieldは終了します。A130 baseの周囲に-30〜+30 ftの離散offset状態を置き、horizontal GRとtypewell GRのraw multi-scale NCCが真の補正状態を識別できるかをStage 21Bの非重複58 wellsで監査します。oracle、top-k、全fold family、固定smooth decoderを測ります。学習もKaggle提出も行いません。CPUランタイムを使用してください。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,os,shutil,subprocess
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir(): subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else: subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None: subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
assert (data_dir/'train').is_dir(),data_dir
def run_checked(command):
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True)
    if result.stdout: print(result.stdout,flush=True)
    if result.returncode:
        print(result.stderr,flush=True); raise RuntimeError(f'command failed: {command}')


## 固定validation split

Stage 21Bの62 cuts・58 wellsだけを使います。offset stateの真値は評価とoracleにだけ使い、NCC emissionとdecoderはsuffix TVTを読みません。


In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
public_oof_run=artifact_dir/'stage7_public_residual_gate_full_v001'
stage21b_run=artifact_dir/'stage21b_prefix_confidence_full_v001'
required=[stage16b_run/'well_assignments.parquet',stage17a_run/'cut_report.parquet',public_oof_run/'base_oof.parquet',stage21b_run/'confidence_cut_report.parquet']
for path in required: assert path.is_file(),path
print(*required,sep='\n')


## Raw emission and fixed decoder audit

各cutでA130 baseを一度生成し、61 offset statesのNCCを計算します。10 cutsごとに進捗を表示します。途中失敗時はこのrunだけを安全に削除します。


In [ ]:
RUN_ID='stage23a_strong_base_ncc_full_v001'; run_dir=artifact_dir/RUN_ID
if run_dir.exists() and not (run_dir/'summary.json').is_file():
    resolved=run_dir.resolve(); expected=(artifact_dir/RUN_ID).resolve()
    assert resolved==expected and resolved.parent==artifact_dir.resolve(),resolved
    print('Removing incomplete prior run:',resolved); shutil.rmtree(resolved)
if not (run_dir/'summary.json').is_file():
    run_checked(['uv','run','rogii-strong-base-ncc','--config','configs/experiment/stage23a_strong_base_ncc.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--public-oof-run',str(public_oof_run),'--validation-run',str(stage21b_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID])
summary=json.loads((run_dir/'summary.json').read_text())
{key:summary[key] for key in ['stage23a_complete','promoted_to_stage23b_learned_emission','cuts','wells','rows','offset_states','offset_coverage','emission_valid_fraction','surface_rmse','oracle_rmse','oracle_delta','median_true_state_rank','top5_recall','top10_recall','random_top10_recall','signal_fold_counts','fraction_signal_count','gates','next_step']}


In [ ]:
import pandas as pd
pd.DataFrame(summary['profile_report']).sort_values('rmse')


最後のsummary辞書とprofile表を共有してください。raw rank信号が全groupで安定した場合だけStage 23B learned emissionへ進みます。decoderの最良profileを後付け採用しません。
